In [1]:
import pysam
import os
import cv2
import numpy as np
import cigar
import sys
import pandas as pd
import torch
import torch.nn as nn
import gc

from torchvision.transforms import transforms as tt
from torch.utils.data import Dataset
from torch.utils.data import DataLoader

In [2]:
SV_TYPES = ['INS', 'DEL', 'INV', 'DUP']
minsvlen = 50
maxsvlen = 10000
dropout = 0.5
minmapq = 20
length = 200
width = 100
line = [900, 128, 128, 1]
model = '/mnt/WD/VGP/VN_007_920_benchmark/sv_deeplearning/cnnLSV/src/realmodel.pt'
dataset = 'real'
mean_std = eval(open('/mnt/WD/VGP/VN_007_920_benchmark/sv_deeplearning/cnnLSV/src/mean_std.txt').read())

In [3]:
sv_info = dict()
for svtype in SV_TYPES:
    sv_info[svtype] = {'minsvlen' : minsvlen,
                        'maxsvlen' : maxsvlen,
                        'model' : model,
                        'dropout' : dropout,
                        'line' : line,
                        'mean' : mean_std[dataset][svtype]['mean'],
                        'std' : mean_std[dataset][svtype]['std']}


In [4]:
def deal_sup(sup_info, min_mapq):
    # [[chr20,167320,+,2436S1758M55I,60,167], [chr20,166350,-,1810S970M40I1429S,60,106]]
    new_sup = []
    for info in sup_info:
        info = info.split(',')
        if int(info[4]) >= min_mapq:
            cigar_list = list(cigar.Cigar(info[3]).items())  # [(2436, 'S'), (1758, 'M'), (55, 'I')]
            ref_start = int(info[1]) - 1
            ref_end = 0
            ori = 0 if info[2] == '-' else 1  # 方向0左1右
            for i in cigar_list:
                if i[1] == 'M':
                    ref_end = ref_start + int(i[0])
            new_sup.append([info[0], ref_start, ref_end, ori])
    return new_sup

def is_diff_ori(info):
    ori = info[0][2]
    for i in info[1:]:
        if i[2] != ori:
            return True
    return False

def discard_del(ref_start, ref_end, cigar, value, sv_size=1):
    normal_list = list()
    start = end = ref_start
    for c in cigar:
        if c[0] in [0, 7, 8]:
            end += c[1]
        elif c[0] == 2:
            if c[1] >= sv_size:
                if end - 1 >= start:
                    normal_list.append([start, end - 1, value])
                start = end + c[1]
                end = start
            else:
                end += c[1]
    if ref_end >= start:
        normal_list.append([start, ref_end, value])
    return normal_list


def encode_indel(bam, chrname, bppair, min_mapq, svtype):
    bp1, bp2 = bppair[0], bppair[1]
    search_range = bp2 - bp1
    s1, _ = bp1 - search_range, bp1
    _, e2 = bp2, bp2 + search_range
    read_base = 0
    ref_base = 3 * search_range
    color_list = list()  # [start, end, value]
    read_dic = dict()
    # 1) analysis cigar
    bam_fetch = bam.fetch(chrname, s1, e2, multiple_iterators=True)
    for read in bam_fetch:
        if read.mapq <= min_mapq:
            continue
        name = read.query_name
        ref_start = read.reference_start
        ref_end = read.reference_end
        query_start = read.query_alignment_start
        query_end = read.query_alignment_end
        if name not in read_dic:
            read_dic[name] = [[query_start, query_end, ref_start, ref_end]]
        else:
            read_dic[name].append([query_start, query_end, ref_start, ref_end])
        cigar = read.cigartuples
        # statistic indel
        shift_ins = 0
        shift_del = 0
        start = ref_start
        ins_list = []
        del_list = []
        normal_list = []
        for c in cigar:
            if c[0] in [0, 2, 7, 8]:
                shift_ins += c[1]
            if c[0] == 1:
                ins_list.append([ref_start + shift_ins, ref_start + shift_ins + c[1] - 1])
            if c[0] in [0, 7, 8]:
                shift_del += c[1]
            if c[0] == 2:
                normal_list.append([start, ref_start + shift_del - 1])
                del_list.append([ref_start + shift_del, ref_start + shift_del + c[1] - 1])
                shift_del += c[1]
                start = ref_start + shift_del
        if ref_end - start >= 0:
            normal_list.append([start, ref_end])
        normal_list = np.array(normal_list)
        sv_list = []

    # 2) analysis split read
        for k, v in read_dic.items():
            if len(v) <= 1:
                continue
            v = sorted(v, key=lambda x: x[0])       # query_start, query_end, ref_start, ref_end
            for i in range(len(v)-1):
                if (v[i+1][0]-v[i][1]) - (v[i+1][2]-v[i][3]) >= 50:
                    ins_len = (v[i+1][0]-v[i][1]) - (v[i+1][2]-v[i][3])
                    ins_list.append([v[i][3], v[i][3]+ins_len])
                elif (v[i+1][2]-v[i][3]) - (v[i+1][0]-v[i][1]) >= 50:
                    del_list.append([v[i][3], v[i+1][2]])
        if svtype.upper() in ['INS']:
            sv_list = ins_list
        elif svtype.upper() in ['DEL']:
            sv_list = del_list
        sv_list = np.array(sv_list)            
        for sv in sv_list:
            s, e = sv
            value = 0
            if e-s <= e2 - s1:
                value += 1
                if bp1 <= (s+e)//2 <= bp2:
                    value += 2
                    if e-s <= 2*(bp2 - bp1):
                        value += 4
                        read_base += e - s + 1
            color_list.append([s, e, value])
        for sv in normal_list:
            s, e = sv
            read_base += max(0, min(e, e2)-max(s, s1))
    color_list = np.array(color_list)
    depth = read_base//ref_base
    return color_list, depth

def encode_inv(bam, chrname, bppair, min_mapq):
    bp1, bp2 = bppair[0], bppair[1]
    search_range = bp2 - bp1  # bp1     bp2
    s1, e1 = bp1 - search_range, bp1  # s1      e1      s2      e2
    s2, e2 = bp2, bp2 + search_range
    read_base = 0
    ref_base = 2 * search_range
    color_list = list()  # [start, end, value]
    ori_state = dict()
    bam_fetch = bam.fetch(chrname, max(bp1 - search_range, 0), bp2 + search_range, multiple_iterators=True)
    for read in bam_fetch:
        read_name = read.query_name
        ref_start = read.reference_start
        ref_end = read.reference_end
        ref_ori = read.is_reverse
        cigar = read.cigartuples
        # value = 0
        # if cigar[0][0] == 4 or cigar[0][0] == 5 or cigar[-1][0] == 4 or cigar[-1][0] == 5:
        #     value += 1
        value = 1
        state = -1
        if read_name not in ori_state:
            tags = read.get_tags()
            is_sa = False
            sup_info = list()
            for tag in tags:
                if tag[0] == 'SA':
                    is_sa = True
                    sup_info = tag[1].split(';')[:-1]
                    break
            if is_sa:
                sup_info = deal_sup(sup_info, min_mapq)
                info = [[chrname, ref_start, ref_end, ref_ori]] + sup_info
                info_new = list()
                for i in info:
                    if i[0] == read.reference_name:
                        if min(e2, i[2]) - max(s1, i[1]) > 0:
                            info_new.append(i)
                diff_ori = is_diff_ori(info_new)
                state = 1 if diff_ori else 0
                ori_state[read_name] = state
        else:
            state = ori_state[read_name]
        if bp1 <= (ref_start + ref_end) // 2 <= bp2:
            value += 2
            if state == 1:
                value += 4
        color_list += discard_del(ref_start, ref_end, cigar, value)
        d1 = min(ref_end, e1) - max(ref_start, s1)
        d2 = min(ref_end, e2) - max(ref_start, s2)
        if d1 > 0:
            read_base += d1
        if d2 > 0:
            read_base += d2
    color_list = np.array(color_list)
    depth = read_base // ref_base
    return color_list, depth

def encode_dup(bam, chrname, bppair, min_mapq):
    bp1, bp2 = bppair[0], bppair[1]
    search_range = bp2 - bp1
    s1, e1 = bp1 - search_range, bp1
    s2, e2 = bp2, bp2 + search_range
    read_base = 0
    ref_base = 3 * search_range
    color_list = list()  # [start, end, value]
    read_dic = dict()
    # 1) analysis cigar
    bam_fetch = bam.fetch(chrname, s1, e2, multiple_iterators=True)
    for read in bam_fetch:
        if read.mapq <= min_mapq:
            continue
        name = read.query_name
        ref_start = read.reference_start
        ref_end = read.reference_end
        query_start = read.query_alignment_start
        query_end = read.query_alignment_end
        if name not in read_dic:
            read_dic[name] = [[query_start, query_end, ref_start, ref_end]]
        else:
            read_dic[name].append([query_start, query_end, ref_start, ref_end])
        cigar = read.cigartuples
        # statistic indel
        shift_ins = 0
        shift_del = 0
        start = ref_start
        ins_list = list()
        del_list = list()
        normal_list = list()
        for c in cigar:
            if c[0] in [0, 2, 7, 8]:
                shift_ins += c[1]
            if c[0] == 1:
                ins_list.append([ref_start + shift_ins, ref_start + shift_ins + c[1] - 1])
            if c[0] in [0, 7, 8]:
                shift_del += c[1]
            if c[0] == 2:
                normal_list.append([start, ref_start + shift_del - 1])
                del_list.append([ref_start + shift_del, ref_start + shift_del + c[1] - 1])
                shift_del += c[1]
                start = ref_start + shift_del
        if ref_end - start >= 0:
            normal_list.append([start, ref_end])
        normal_list = np.array(normal_list)
        dup_list = list()

        # analysis dup
        for k, v in read_dic.items():
            if len(v) <= 1:
                continue
            v = sorted(v, key=lambda x: x[0])  # query_start, query_end, ref_start, ref_end
            for i in range(len(v) - 1):
                if min(v[i+1][3], v[i][3]) - max(v[i+1][2], v[i][2]) >= 0:
                    dup_list += [[v[i][2], v[i][3]], [v[i+1][2], v[i+1][3]]]
                # if (v[i+1][0]-v[i][1]) - (v[i+1][2]-v[i][3]) >= 50:
                    # ins_len = (v[i+1][0]-v[i][1]) - (v[i+1][2]-v[i][3])
                    # ins_list.append([v[i][2], v[i][2]+ins_len])
        for sv in dup_list+ins_list:
            s, e = sv
            value = 0
            if e - s <= e2 - s1:
                value += 1
                if bp1 <= (s + e) // 2 <= bp2:
                    value += 2
                    if e - s <= 2 * (bp2 - bp1):
                        value += 4
                        read_base += e - s + 1
            color_list.append([s, e, value])
        for sv in normal_list:
            s, e = sv
            read_base += max(0, min(e, e2) - max(s, s1))
    color_list = np.array(color_list)
    depth = read_base // ref_base
    return color_list, depth

def shift_index_indel(color_list, bppair):
    s1, e1 = bppair[0], bppair[1]           #s2     s1      e1      e2
    sv_len = e1 - s1 + 1
    s2 = s1 - sv_len
    e2 = e1 + sv_len
    shift = list()
    for read in color_list:
        start, end, color = read
        if min(end, e2) - max(start, s2) <= 0:
            continue
        shift_start = max(start, s2) - s2
        shift_end = min(end, e2) - s2
        shift.append([shift_start, shift_end, color])
    return shift, sv_len*3

def shift_index_inv(color_list, bppair):
    threshold = 1000
    bp1, bp2 = bppair[0:2]
    shift = list()
    if bp2 - bp1 <= threshold:
        threshold = bp2 - bp1
        s, e = bp1 - threshold, bp2 + threshold
        for read in color_list:
            start, end, color = read
            if min(end, e) - max(start, s) <= 0:
                continue
            shift_start = max(start, s) - s
            shift_end = min(end, e) - s
            shift.append([shift_start, shift_end, color])
    else:
        s1, e1 = bp1 - threshold, bp1 + threshold // 2
        s2, e2 = bp2 - threshold // 2, bp2 + threshold
        for read in color_list:
            start, end, color = read
            if min(end, e1) - max(start, s1) > 0:
                shift_start1 = max(start, s1) - s1
                shift_end1 = min(end, e1) - s1
                shift.append([shift_start1, shift_end1, color])
            if min(end, e2) - max(start, s2) > 0:
                shift_start2 = max(start, s2) - s2 + threshold//2*3
                shift_end2 = min(end, e2) - s2 + threshold//2*3
                shift.append([shift_start2, shift_end2, color])
    return shift, threshold*3


def deal_draw_list(draw_list, draw_range):
    dl1 = list()
    dl2 = list()
    bound = (draw_range - 1) // 2
    for draw in draw_list:
        left, right, value = draw
        if 0 <= right <= bound:
            dl1.append(draw)
        elif bound + 1 <= left <= draw_range - 1:
            dl2.append([left - bound - 1, right - bound - 1, value])
        else:
            dl1.append([left, bound, value])
            dl2.append([0, right - bound - 1, value])
    return dl1, dl2

def generate_img(elelist, col, depth):
    imageback = np.zeros((len(elelist), col), dtype=np.uint8)
    index = 0
    for read in elelist:
        start, end, value = read
        imageback[index][start: end] = value
        index += 1
    imageback = np.sort(imageback, axis=0)[::-1]
    imageback = imageback[0:depth, :].tolist()  # depth = 是计算出来的
    color_map = np.array([[0, 0, 0], [255, 0, 0], [0, 255, 0], [255, 255, 0],
                        [0, 0, 255], [255, 0, 255], [0, 255, 255], [255, 255, 255]])
    imageback = color_map[imageback]

    imageback = np.array(imageback, dtype=np.uint8)
    return imageback

def draw_image(draw_list, draw_range, depth, length, width):
    bound = (draw_range-1)//2
    dl1, dl2 = deal_draw_list(draw_list, draw_range)
    img1 = generate_img(dl1, bound+1, depth)
    img2 = generate_img(dl2, draw_range-bound-1, depth)
    # return img1, img2
    image = np.append(img1, img2, axis=1)
    image = cv2.resize(image, (length, width), interpolation=cv2.INTER_CUBIC)
    return image

In [5]:
def getProperty(x, keys):
    try:
        tmp = x.split(";")
        for i in tmp:
            if keys in i:
                return i.split("=")[-1]
    except:
        return None

class SV(object):
    """
    Constructs a structural variant (SV) candidate
    """
    def __init__(self, variants, index):
        self.index = index
        self.chromosome = variants[0]
        self.begin = int(variants[1])
        self.ref = variants[3]
        self.alt = variants[4]
        self.filter = variants[6]
        self.info = variants[7]
        self.type = getProperty(self.info, keys='SVTYPE')
        self.length = int(getProperty(self.info, keys='SVLEN'))
        self.end = int(getProperty(self.info, keys='END'))
        if (self.type == 'DEL') & (self.end is None):
            self.end = self.begin + self.length

In [6]:
def read_vcf(vcf_record):
    header = []
    sv_lists = []

    with open(vcf_record, 'r') as fp:
        lines = fp.readlines()
    for index, line in enumerate(lines):
        if '#' in line:
            header.append(str(index))
            continue

        items = line.split("\t")
        sv_type = getProperty(items[7], keys='SVTYPE')
        if sv_type in SV_TYPES:
            sv = SV(items, index)
            sv_lists.append(sv)
    return header, sv_lists

class StructuralVariant(Dataset):
    def __init__(self, vcf_path, bam_path):
        self.header, self.sv_lists = read_vcf(vcf_path)
        self.bam = pysam.AlignmentFile(bam_path, 'rb')

    def __len__(self):
        return len(self.sv_lists)
    
    def __getitem__(self, idx):
        try:
            sv = self.sv_lists[idx]
            chrname, start, end, svtype, info = sv.chromosome, sv.begin, sv.end, sv.type, sv_info[sv.type]
            sv_index = sv.index
            bp = [start, end]
            draw_list = list()
            draw_range, depth = 0, 0
            if svtype == 'DEL' or svtype == 'INS':
                color_list, depth = encode_indel(self.bam, chrname, bp, minmapq, svtype)
                draw_list, draw_range = shift_index_indel(color_list, bp)  # del没有width
            elif svtype == 'DUP':
                color_list, depth = encode_dup(self.bam, chrname, bp, minmapq)
                draw_list, draw_range = shift_index_indel(color_list, bp)
            elif svtype == 'INV':
                color_list, depth = encode_inv(self.bam, chrname, bp, minmapq)
                draw_list, draw_range = shift_index_inv(color_list, bp)
            img = draw_image(draw_list, draw_range, depth, length, width)
            if img is not None:
                return img, svtype, sv_index
        except Exception as e:
            return np.zeros((width, length, 3)), svtype, sv_index

In [7]:
class Net(nn.Module):
    def __init__(self, line, dropout):
        l1_in, l1_out, l2_in, l2_out = line
        super(Net, self).__init__()
        self.conv = nn.Sequential(nn.Conv2d(3, 3, 3, 1, padding=1), nn.ReLU(), nn.MaxPool2d(2),  # 3*100*50
                                  nn.Conv2d(3, 3, 3, 1, padding=1), nn.ReLU(), nn.MaxPool2d(2),  # 3*50*25
                                  nn.Conv2d(3, 3, 3, 1, padding=1), nn.ReLU(), nn.MaxPool2d(2))  # 3*25*12
        self.linear = nn.Sequential(nn.Linear(l1_in, l1_out), nn.ReLU(), nn.Dropout(dropout), nn.Linear(l2_in, l2_out))
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        x = self.conv(x)
        x = x.view(x.size(0), -1)
        x = self.linear(x)
        return self.sigmoid(x)


def pre_sv(img, net_model_file_path, means, std, line, dropout, svtype):
    trans = tt.Compose([tt.ToTensor(), tt.Normalize(means, std)])
    net = Net(line, dropout)
    net.load_state_dict(torch.load(net_model_file_path, map_location=torch.device('cuda'), weights_only=True)[svtype])
    img = trans(img)
    x = torch.unsqueeze(img, 0)
    pre = net(x).item()
    del net
    gc.collect() 
    if pre >= 0.5:
        return 1
    else:
        return 0

# INFERENCE

In [8]:
vcf_path = '/mnt/WD/VGP/VN_007_920_benchmark/sniffles2/VN007.merge.hifi_reads_pacbio.ngmlr.sorted.vcf'
bam_path = '/mnt/WD/VGP/alignment/VN007/VN007.merge.bam'
batch_size = 512
svDataset = StructuralVariant(vcf_path, bam_path)
dataloader = DataLoader(svDataset, batch_size=batch_size, shuffle=False)

In [9]:
def get_device(gpu=False):
    if gpu:
        device = f"cuda:{gpu}" 
    else: 
        device = 'cpu'
    return device

device = get_device(0)
success = []

with torch.no_grad():
    for batch, (list_img, svtype_lists, sv_indexes) in enumerate(dataloader):
        for index in range(len(list_img)):
            img = list_img[index]
            if torch.all(img==0):
                continue
            sv_index = sv_indexes[index]
            svtype = svtype_lists[index]
            info = sv_info[svtype]
            minlen, maxlen, pt, dropout, line, mean, std = info['minsvlen'], info['maxsvlen'], info['model'], info['dropout'], info['line'], info['mean'], info['std']
            img = np.array(img, dtype=np.float32)
            pre = pre_sv(img, model, mean, std, line, dropout, svtype)
            if pre:
                success.append(str(sv_index))
            with open(f'tmp_dl/result_sv_batch_{batch}_0508.txt', 'w') as fp:
                fp.write("\n".join(success))

KeyboardInterrupt: 

In [ ]:
success

# DEBUG

In [ ]:
header, sv_lists = read_vcf(vcf_path)
bam = pysam.AlignmentFile(bam_path)


In [ ]:
fail = []
success = []
print("Start with len=",len(sv_lists))
for i in range(len(sv_lists)):
    try:
        sv = sv_lists[i]
        chrname, start, end, svtype, info = sv.chromosome, sv.begin, sv.end, sv.type, sv_info[sv.type]
        bp = [start, end]
        svlen = end - start
        minlen, maxlen, pt, dropout, line, mean, std = info['minsvlen'], info['maxsvlen'], info['model'], info['dropout'], info['line'], info['mean'], info['std']
        draw_list = list()
        draw_range, depth = 0, 0
        if svtype == 'DEL' or svtype == 'INS':
            color_list, depth = encode_indel(bam, chrname, bp, minmapq, svtype)
            draw_list, draw_range = shift_index_indel(color_list, bp)  # del没有width
        if svtype == 'DEL' or svtype == 'INS':
            color_list, depth = encode_indel(bam, chrname, bp, minmapq, svtype)
            draw_list, draw_range = shift_index_indel(color_list, bp)  # del没有width
        elif svtype == 'DUP':
            color_list, depth = encode_dup(bam, chrname, bp, minmapq)
            draw_list, draw_range = shift_index_indel(color_list, bp)
        elif svtype == 'INV':
            color_list, depth = encode_inv(bam, chrname, bp, minmapq)
            draw_list, draw_range = shift_index_inv(color_list, bp)
        img = draw_image(draw_list, draw_range, depth, length, width)
        if img is not None:
            print(i)
            success.append(i)
    except Exception as e:
        fail.append(i)

Start with len= 26647
2


8
13
14
19
20
21
24
32
33
35
42
46
50
51
55
59
62
72
73
76
95
97
98
102
109
118
121
132
140
146
153
159
163
181
183
184
190
191
195
204
219
220
221
246
248
256
258
266
267
268
279
287
288
296
299
308
310
312
315
316
319
324
327
330
331
333
334
335
340
341
352
353
354
358
362
363
364
375
377
379
381
392
395
397
403
421
423
424
425
427
428
435
439
440
444
463
465
467
473
474
475
477
478
483
484
486
500
503
511
514
516
529
531
532
537
540
551
554
559
560
571
572
580
583
589
599
601
602
609
612
613
614
615
618
625
628
630
631
632
637
646
654
657
660
663
667
670
673
679
686
689
690
691
692
698
701
703
707
708
710
713
715
720
722
724


KeyboardInterrupt: 

In [ ]:
success

[]